# DOC-01 CASA form test (rules + PaddleOCR-VL + evidence check)

Upload `casa_form.py`, `casa_reader.py` and `validation_rules.py` (from `notebooks/doc01_support/`) and your CASA JPG/PNG files. Turn on the GPU first (Runtime > Change runtime type > T4).

The run writes `doc01_results.json` for download.

In [ ]:
# Do NOT upgrade torch or numpy: that breaks the preinstalled torch on Colab.
!pip install -q rapidocr_onnxruntime accelerate
import transformers, torch
print('transformers', transformers.__version__, '| CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''  )

In [ ]:
import os, sys, glob
WORK='/content/doc01'
os.makedirs(WORK, exist_ok=True)
from google.colab import files
print('Upload casa_form.py, casa_reader.py, validation_rules.py and your CASA images')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(os.path.join(WORK, name), 'wb') as f:
        f.write(data)
sys.path.insert(0, WORK)
images = sorted(p for ext in ('*.jpg','*.jpeg','*.png') for p in glob.glob(os.path.join(WORK, ext)))
print(len(images), 'images:', [os.path.basename(p) for p in images])

In [ ]:
import os, sys, json, time
WORK='/content/doc01'
missing=[f for f in ('casa_form.py','casa_reader.py','validation_rules.py') if not os.path.exists(os.path.join(WORK,f))]
if missing:
    raise SystemExit('Missing files in ' + WORK + ': ' + ', '.join(missing) + '. Run the upload cell first and upload them.')
sys.path.insert(0, WORK)
from PIL import Image
from casa_reader import PaddleVLCasaReader, read_casa_form
import glob
images = sorted(p for ext in ('*.jpg','*.jpeg','*.png') for p in glob.glob(os.path.join(WORK, ext)))
if not images: raise SystemExit('No CASA images found in ' + WORK + '. Run the upload cell and upload them.')
t=time.time()
reader = PaddleVLCasaReader()
reader._load()
print('model loaded in', round(time.time()-t,1), 's')
results = {}
for p in images:
    r = read_casa_form(Image.open(p), reader)
    results[os.path.basename(p)] = r
    print('==', os.path.basename(p), '| status:', r['status'], '| model secs:', r['timing']['model_seconds'])
    for k, v in r['fields'].items():
        mark = 'REVIEW' if v['review_required'] else 'ok'
        print(f"   {k:26s} {str(v['value']):28s} {v['null_reason'] or '':9s} {mark:6s} {v['error'] or ''}")
    print()

In [ ]:
json.dump(results, open('/content/doc01/doc01_results.json','w'), indent=2, ensure_ascii=False, default=str)
files.download('/content/doc01/doc01_results.json')